<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Agentes Basados en Conocimiento: Lógica Proposicional 🧠🔗 🧸
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.1em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Introducción a la Inteligencia Artificial — Taller Práctico Evaluativo
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 05 • Para Dummies
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

---
## 🎈 ¿Qué vamos a hacer en este taller?

1. 🕵️ **Resolver un mini acertijo de "veo, veo" lógico** con 2 personajes que dicen la verdad o mienten siempre.
2. 🔄 **Traducir 2 fórmulas** a un formato especial llamado CNF, y comprobarlo con código.
3. 🎨 **Inventar tu propio acertijo** con al menos 2 pistas.

> ⚠️ **Instrucciones de entrega:** completa cada celda marcada con `# TODO:`. No se aceptan notebooks con celdas sin ejecutar o con errores. Antes de entregar, usa *Restart Kernel and Run All Cells* para verificar que todo el notebook corre de principio a fin.

---
### 📦 Nuestra Calculadora de Lógica (ya resuelta, no la modifiques)

Piensa en este bloque de código como una calculadora que entiende frases lógicas ("Y", "O", "NO", "SI...ENTONCES", "SI Y SOLO SI") y que puede decirte si una conclusión es **siempre verdadera** dado lo que ya sabes (eso es "model_check").

In [ ]:
import itertools


class Symbol:
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return self.name
    def __hash__(self):
        return hash(("symbol", self.name))
    def __eq__(self, other):
        return isinstance(other, Symbol) and self.name == other.name
    def evaluate(self, model):
        try:
            return model[self.name]
        except KeyError:
            raise Exception(f"La variable {self.name} no está definida en el modelo")
    def symbols(self):
        return {self.name}


class Not:
    def __init__(self, operand):
        self.operand = operand
    def __repr__(self):
        return f"¬({self.operand})"
    def evaluate(self, model):
        return not self.operand.evaluate(model)
    def symbols(self):
        return self.operand.symbols()


class And:
    def __init__(self, *conjuncts):
        self.conjuncts = list(conjuncts)
    def __repr__(self):
        return "(" + " ∧ ".join(str(c) for c in self.conjuncts) + ")"
    def evaluate(self, model):
        return all(c.evaluate(model) for c in self.conjuncts)
    def symbols(self):
        return set.union(*[c.symbols() for c in self.conjuncts]) if self.conjuncts else set()


class Or:
    def __init__(self, *disjuncts):
        self.disjuncts = list(disjuncts)
    def __repr__(self):
        return "(" + " ∨ ".join(str(d) for d in self.disjuncts) + ")"
    def evaluate(self, model):
        return any(d.evaluate(model) for d in self.disjuncts)
    def symbols(self):
        return set.union(*[d.symbols() for d in self.disjuncts]) if self.disjuncts else set()


class Implication:
    def __init__(self, antecedent, consequent):
        self.antecedent = antecedent
        self.consequent = consequent
    def __repr__(self):
        return f"({self.antecedent} → {self.consequent})"
    def evaluate(self, model):
        return (not self.antecedent.evaluate(model)) or self.consequent.evaluate(model)
    def symbols(self):
        return self.antecedent.symbols() | self.consequent.symbols()


class Biconditional:
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def __repr__(self):
        return f"({self.left} ↔ {self.right})"
    def evaluate(self, model):
        return self.left.evaluate(model) == self.right.evaluate(model)
    def symbols(self):
        return self.left.symbols() | self.right.symbols()


def model_check(knowledge, query):
    """Verifica si `knowledge` implica lógicamente `query` (entailment),
    probando todos los modelos posibles (fuerza bruta) sobre los símbolos
    involucrados en ambas fórmulas."""
    simbolos = knowledge.symbols() | query.symbols()

    def check_all(pendientes, model):
        if not pendientes:
            if knowledge.evaluate(model):
                return query.evaluate(model)
            return True
        resto = pendientes.copy()
        s = resto.pop()
        modelo_true = model.copy(); modelo_true[s] = True
        modelo_false = model.copy(); modelo_false[s] = False
        return check_all(resto, modelo_true) and check_all(resto, modelo_false)

    return check_all(simbolos, {})


def formulas_equivalentes(f1, f2):
    """Devuelve True si f1 y f2 son lógicamente equivalentes (mismo valor de
    verdad en TODOS los modelos posibles). Útil para verificar una conversión a CNF."""
    simbolos = list(f1.symbols() | f2.symbols())
    for valores in itertools.product([True, False], repeat=len(simbolos)):
        modelo = dict(zip(simbolos, valores))
        if f1.evaluate(modelo) != f2.evaluate(modelo):
            return False
    return True


print("Mini-librería de lógica proposicional cargada correctamente ✅")

---
### 📌 Reto 1: Caballeros y Escuderos (versión mini) 🧸

Solo 2 personajes esta vez:
- **Dani dice:** "Yo soy Escudero." (¡Pista: esta frase, si la dice un Caballero, sería una mentira imposible! Piénsalo bien.)
- **Ely dice:** "Dani y yo somos del mismo tipo."

**Ejercicio 1.1:** completa `conocimiento` siguiendo la pista de los comentarios y usa `model_check` para saber qué es cada uno.

In [ ]:
Dani_Caballero = Symbol("Dani_Caballero")
Ely_Caballero = Symbol("Ely_Caballero")

# TODO: la frase de Dani ("Yo soy Escudero", es decir Not(Dani_Caballero)) es
# verdadera si y solo si Dani es Caballero:
# Biconditional(Not(Dani_Caballero), Dani_Caballero)
clausula_dani = None  # TODO

# TODO: la frase de Ely ("Dani y yo somos del mismo tipo") es verdadera si y
# solo si Ely es Caballero:
# Biconditional(Biconditional(Dani_Caballero, Ely_Caballero), Ely_Caballero)
clausula_ely = None  # TODO

conocimiento = And(clausula_dani, clausula_ely)  # TODO: revisa que clausula_dani y clausula_ely ya estén definidas

for nombre, simbolo in [("Dani", Dani_Caballero), ("Ely", Ely_Caballero)]:
    es_caballero = None  # TODO: usa model_check(conocimiento, simbolo)
    print(f"{nombre} es Caballero:", es_caballero)

---
### 📌 Reto 2: Traducir Fórmulas a CNF 🧸

CNF es solo una forma "ordenada" de escribir una fórmula usando nada más `And`, `Or` y `Not` (sin flechas `→` ni `↔`).

**Ejemplo ya resuelto:** `¬(P ∧ Q)` se convierte, aplicando la ley de De Morgan, en `(¬P ∨ ¬Q)`.

```python
f_ejemplo_original = Not(And(P, Q))
f_ejemplo_cnf = Or(Not(P), Not(Q))
# formulas_equivalentes(f_ejemplo_original, f_ejemplo_cnf) -> True
```

**Ejercicio 2.1:** convierte a mano estas 2 fórmulas y luego constrúyelas en código:
1. `F1 = P → Q` (pista: `A → B` equivale a `¬A ∨ B`)
2. `F2 = (P ↔ Q)` (pista: primero conviértela en `(P → Q) ∧ (Q → P)`, y luego cada implicación en una disyunción)

In [ ]:
P = Symbol("P")
Q = Symbol("Q")

f1_original = Implication(P, Q)     # P → Q
f2_original = Biconditional(P, Q)   # (P ↔ Q)

# TODO: construye la versión CNF de F1 (solo Or/Not)
f1_cnf = None  # TODO

# TODO: construye la versión CNF de F2 (And de dos Or, solo con Or/Not/And)
f2_cnf = None  # TODO

print("F1 original ≡ F1 CNF:", formulas_equivalentes(f1_original, f1_cnf))
print("F2 original ≡ F2 CNF:", formulas_equivalentes(f2_original, f2_cnf))

---
### 📌 Reto 3: Inventa tu Propio Mini-Acertijo 🧸

**Ejercicio 3.1:** inventa un acertijo simple con **al menos 2 pistas** (por ejemplo: "¿quién se comió la galleta?", con 2 sospechosos). Sigue el patrón del Reto 1.

In [ ]:
# TODO 1: describe tu acertijo en un comentario (enunciado + al menos 2 pistas)

# TODO 2: define los símbolos que necesites, ej:
# Sospechoso_A = Symbol("Sospechoso_A")

# TODO 3: construye mi_conocimiento como un And(...) con tus pistas (usa Biconditional
# o Implication según lo que necesites, igual que en el Reto 1)
mi_conocimiento = None  # TODO

# TODO 4: usa model_check para responder alguna pregunta sobre tu acertijo
respuesta = None  # TODO
print("Respuesta a mi acertijo:", respuesta)

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Introducción a la Inteligencia Artificial</i><br>
    Taller Para Dummies — Módulo 05: Agentes Basados en Conocimiento
  </p>
</div>